In [2]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)

2.11.0+cu126
True
12.6


#### Define the classification model. 


1. 3 cnn layers start by 3 colors r,g,b outputing 16 channels and fiishing with 64 channels. size of kernel scanning (3x3 with a 1 pixel padding of zeros.)

2. 3 fully connected layers 256/2 = 128 128/2 = 64 64/2 = 32 (maxpool operations)  so 64 cnn channels * 32 x 32, 256 (pixel of images)

3. final fully connected layer output = number of classes (2)

4. after each conv or fully connected layer a relu activation function is applied


In [5]:
class ClassifierModel(nn.Module):

  def __init__(self, num_classes=2):
    super().__init__()

    self.conv1 = nn.Conv2d(in_channels= 3, out_channels= 16, kernel_size= 3, padding= 1)
    self.conv2 = nn.Conv2d(in_channels= 16, out_channels= 32, kernel_size= 3, padding= 1)
    self.conv3 = nn.Conv2d(in_channels= 32, out_channels=64, kernel_size= 3, padding=1)

    self.pool = nn.MaxPool2d(kernel_size= 2, stride= 2)
    self.dropout = nn.Dropout(0.5) # added due to overfitting, lowered to reduce the aggresiveness

    # 64 output channles from cnv3 * 32 * 32 pixel images from the maxpools 3 times /2
    self.fc1 = nn.Linear(64 * 32 * 32, 256)
    self.fc2 = nn.Linear(256, 128)
    self.fc3 = nn.Linear(128, num_classes)


  def forward(self, x):

      x = self.conv1(x)
      x = F.relu(x)
      x= self.pool(x)

      x = self.conv2(x)
      x = F.relu(x)
      x = self.pool(x)

      x= self.conv3(x)
      x = F.relu(x)
      x = self.pool(x)

      x = x.view(-1, 64 * 32 * 32)

      x = self.fc1(x)
      x = F.relu(x)
      x = self.dropout(x)

      x = self.fc2(x)
      x = F.relu(x)
      x = self.dropout(x)

      x = self.fc3(x)

      return x

### Define the model and pass the number of classes. (2)

In [12]:
model = ClassifierModel(num_classes=2)
print(model)

ClassifierModel(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=65536, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=2, bias=True)
)


## Old dataset with 800 images 



#### Load and Pre-process the data

1. defeine the transform of the images ((although they are all 256x256) resize to 256x256 pixels, convert arrays to tensors, and normalize values from -1.0 to 1.0)

2. Load th edata using datasets.ImageFolder passign the path to the data and the transform we created.

3. Break the dataset into 80% train data and 20% evaluation data. (randomly)

4. Create 2 data loaders one for trainingandone for evaluation with batch of 32


In [4]:
import os

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])


data_path = os.path.join(os.getcwd(), "training")
dataset = datasets.ImageFolder(root=data_path, transform=transform)

train_size = int(0.8 * len(dataset))
eval_size = len(dataset) - train_size
train_dataset, eval_dataset = random_split(dataset, [train_size, eval_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False)

print(f"Number of training samples: {len(train_dataset)}")  
print(f"Number of evaluation samples: {len(eval_dataset)}")


Number of training samples: 640
Number of evaluation samples: 160


#### Load and Pre-process the data

1. defeine the transform of the images ((although they are all 256x256) resize to 256x256 pixels, convert arrays to tensors, and normalize values from -1.0 to 1.0)

2. Load th edata using datasets.ImageFolder passign the path to the data and the transform we created.

3. Break the dataset into 80% train data and 20% evaluation data. (randomly)

4. Create 2 data loaders one for trainingandone for evaluation with batch of 32


In [16]:
# Test wether the model with the full dataset

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

data_path = os.path.join(os.getcwd(), "archive/PetImages")
dataset = datasets.ImageFolder(root=data_path, transform=transform)

train_size = int(0.8 * len(dataset))
eval_size = len(dataset) - train_size
train_dataset, eval_dataset = random_split(dataset, [train_size, eval_size])

train_loader = DataLoader(train_dataset, batch_size=32, num_workers=4, shuffle=True)
eval_loader = DataLoader(eval_dataset, batch_size=32, num_workers=4, shuffle=False)

print(f"Number of training samples: {len(train_dataset)}")  
print(f"Number of evaluation samples: {len(eval_dataset)}")


Number of training samples: 19998
Number of evaluation samples: 5000


## Load the device to the GPU if available

In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

model = model.to(device)

cuda


## Train the model


1. Training for 20 epochs

2. Define the loss criterior to measrue how wrong the prediction was

3. dam optimizer with learnign rate of 0.001 toreduce the loss

4. Trains the model, clears gradients, makes predicitons, computes gradients and updates weights while accumilating loss to average 




In [18]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 20

for epoch in range(num_epochs):

    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()



    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        
        for images, labels in eval_loader:
            images, labels = images.to(device), labels.to(device)
            predicted = model(images).argmax(dim=1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = running_loss / len(train_loader)
    acc = 100 * correct / total
    
    print(f"Epoch {epoch+1}/{num_epochs}  Loss: {avg_loss:.4f}  Eval Acc: {acc:.2f}%")


torch.save(model, "model.pth")

Epoch 1/20  Loss: 0.5315  Eval Acc: 78.98%
Epoch 2/20  Loss: 0.4437  Eval Acc: 81.70%
Epoch 3/20  Loss: 0.3535  Eval Acc: 82.10%
Epoch 4/20  Loss: 0.2485  Eval Acc: 80.66%
Epoch 5/20  Loss: 0.1584  Eval Acc: 80.96%
Epoch 6/20  Loss: 0.1047  Eval Acc: 80.84%
Epoch 7/20  Loss: 0.0852  Eval Acc: 80.36%
Epoch 8/20  Loss: 0.0698  Eval Acc: 79.72%
Epoch 9/20  Loss: 0.0544  Eval Acc: 80.28%
Epoch 10/20  Loss: 0.0530  Eval Acc: 80.94%
Epoch 11/20  Loss: 0.0466  Eval Acc: 79.90%
Epoch 12/20  Loss: 0.0442  Eval Acc: 80.76%
Epoch 13/20  Loss: 0.0396  Eval Acc: 79.90%
Epoch 14/20  Loss: 0.0433  Eval Acc: 81.36%
Epoch 15/20  Loss: 0.0407  Eval Acc: 80.40%
Epoch 16/20  Loss: 0.0357  Eval Acc: 79.72%
Epoch 17/20  Loss: 0.0372  Eval Acc: 80.74%
Epoch 18/20  Loss: 0.0357  Eval Acc: 80.92%
Epoch 19/20  Loss: 0.0325  Eval Acc: 80.98%
Epoch 20/20  Loss: 0.0380  Eval Acc: 80.64%


### Save the model.

In [9]:
torch.save(model, "model.pth")

# Get the model and create a small UI using gradio.

In [ ]:
import gradio as gr

model = torch.load("model.pth", map_location=device, weights_only=False)
model.to(device)
model.eval()

classes = ["Cat", "Dog"]

predict_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

def predict(image):
    model.eval()
    tensor = predict_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = F.softmax(model(tensor), dim=1)[0]
    return {classes[i]: float(probs[i]) for i in range(len(classes))}

gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="Upload image"),
    outputs=gr.Label(num_top_classes=2, label="Prediction"),
    title="Cat vs Dog Classifier (Own Model)"
).launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### Transfer learning using resnet18 - IMAGENET1K_V1

In [199]:
from torchvision import models

model = models.resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(model.fc.in_features, 2)




Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\johnb/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


## Introduced variation in training data with random flips, rorations and colorjutter.

In [200]:
import os

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

data_path = os.path.join(os.getcwd(), "training")
dataset = datasets.ImageFolder(root=data_path, transform=transform)

train_size = int(0.8 * len(dataset))
eval_size = len(dataset) - train_size
train_dataset, eval_dataset = random_split(dataset, [train_size, eval_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of evaluation samples: {len(eval_dataset)}")


Number of training samples: 640
Number of evaluation samples: 160


In [201]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model = model.to(device)


cuda


In [203]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)  

num_epochs = 10  

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in eval_loader:
            images, labels = images.to(device), labels.to(device)
            predicted = model(images).argmax(dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = running_loss / len(train_loader)
    acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{num_epochs}  Loss: {avg_loss:.4f}  Eval Acc: {acc:.2f}%")

print("Training complete.")


Epoch 1/10  Loss: 0.0055  Eval Acc: 90.62%
Epoch 2/10  Loss: 0.0065  Eval Acc: 95.62%
Epoch 3/10  Loss: 0.0237  Eval Acc: 95.00%
Epoch 4/10  Loss: 0.0071  Eval Acc: 95.00%
Epoch 5/10  Loss: 0.0098  Eval Acc: 96.25%
Epoch 6/10  Loss: 0.0067  Eval Acc: 91.25%
Epoch 7/10  Loss: 0.0208  Eval Acc: 95.00%
Epoch 8/10  Loss: 0.0187  Eval Acc: 94.38%
Epoch 9/10  Loss: 0.0040  Eval Acc: 95.62%
Epoch 10/10  Loss: 0.0043  Eval Acc: 96.25%
Training complete.


In [3]:
import gradio as gr

classes = ["Cat", "Dog"]

predict_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

def predict(image):
    model.eval()
    tensor = predict_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = F.softmax(model(tensor), dim=1)[0]
    return {classes[i]: float(probs[i]) for i in range(len(classes))}

gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="Upload image"),
    outputs=gr.Label(num_top_classes=2, label="Prediction"),
    title="Cat vs Dog Classifier (ResNet18)"
).launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\johnb\Desktop\Work_project_AI\venv\Lib\site-packages\gradio\queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\johnb\Desktop\Work_project_AI\venv\Lib\site-packages\gradio\route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\johnb\Desktop\Work_project_AI\venv\Lib\site-packages\gradio\blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\johnb\Desktop\Work_project_AI\venv\Lib\site-packages\gradio\blocks.py", line 1636, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\johnb\Desktop\Work_project_AI\venv\Lib\site-packages\anyio\to_thr

In [11]:
gr.close_all()

Closing server running on port: 7860
